# Customer Feature Engineering

This notebook transforms the cleaned Online Retail II transaction data into
customer-level behavioural features for customer segmentation and business
profiling.

The cleaned transaction dataset retains valid merchandise purchases and
identifiable cancellations separately. Feature construction therefore
distinguishes purchase behaviour from cancellation behaviour rather than
treating all transaction lines identically.

The resulting feature table represents customer behaviour across the complete
observation period and is used for the segmentation analysis in the following
notebook.

The later 90-day repeat-purchase propensity analysis uses a separate
point-in-time feature-engineering process based on an earlier historical cutoff
to prevent future information from leaking into the predictive model.

No clustering or supervised modelling is performed in this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests


REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

In [2]:
LOCAL_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "online_retail_clean.parquet"
)

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "apostolis-bloutsos-data/"
    "customer-segmentation-purchase-propensity/"
    "main/data/processed/online_retail_clean.parquet"
)

if LOCAL_DATA_PATH.exists():
    df = pd.read_parquet(LOCAL_DATA_PATH)
    print("Loaded local cleaned dataset.")

else:
    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()

    LOCAL_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOCAL_DATA_PATH.write_bytes(response.content)

    df = pd.read_parquet(LOCAL_DATA_PATH)
    print("Downloaded cleaned dataset from GitHub.")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Downloaded cleaned dataset from GitHub.
Rows: 820,506
Columns: 14


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 820506 entries, 0 to 820505
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   invoice             820506 non-null  string        
 1   stock_code          820506 non-null  string        
 2   description         820506 non-null  string        
 3   quantity            820506 non-null  int64         
 4   invoice_date        820506 non-null  datetime64[ns]
 5   price               820506 non-null  float64       
 6   customer_id         820506 non-null  string        
 7   country             820506 non-null  string        
 8   source_sheet        820506 non-null  string        
 9   line_revenue        820506 non-null  float64       
 10  is_non_merchandise  820506 non-null  bool          
 11  is_cancellation     820506 non-null  boolean       
 12  is_purchase         820506 non-null  boolean       
 13  transaction_type    820506 no

In [4]:
print(df["transaction_type"].value_counts())

print(
    "Date range:",
    df["invoice_date"].min(),
    "to",
    df["invoice_date"].max()
)

print(f"Customers: {df['customer_id'].nunique():,}")
print(f"Invoices: {df['invoice'].nunique():,}")

transaction_type
purchase        802573
cancellation     17933
Name: count, dtype: int64
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Customers: 5,875
Invoices: 43,877


In [5]:
# Separate purchase and cancellation behaviour for feature construction.

purchases = df[
    df["transaction_type"] == "purchase"
].copy()

cancellations = df[
    df["transaction_type"] == "cancellation"
].copy()

In [6]:
print(f"Purchase rows: {len(purchases):,}")
print(f"Cancellation rows: {len(cancellations):,}")

Purchase rows: 802,573
Cancellation rows: 17,933


## Temporal framework

The project contains two subsequent analytical objectives with different
time requirements.

For customer segmentation, the complete available purchase history is used.
Because segmentation is descriptive rather than predictive, customer behaviour
can be summarized across the full observation period. Recency is measured
relative to one day after the final purchase date in the dataset.

The repeat-purchase propensity model requires a different temporal design.
Customer features must be calculated only from information available up to a
defined historical cutoff date. Purchases occurring during the following
90 days are reserved exclusively for construction of the future
repeat-purchase target.

This distinction prevents future information from leaking into the predictors
used by the classification model.

Accordingly, this notebook first constructs a full-history customer feature
table for segmentation. A separate historical snapshot will later be created
for the 90-day propensity task using the same feature-engineering principles
but an explicit cutoff date.

In [7]:
first_purchase_date = purchases["invoice_date"].min()
last_purchase_date = purchases["invoice_date"].max()

print(f"First purchase: {first_purchase_date}")
print(f"Last purchase:  {last_purchase_date}")

First purchase: 2009-12-01 07:45:00
Last purchase:  2011-12-09 12:50:00


In [8]:
# Defines the segmentation reference date

segmentation_reference_date = (
    last_purchase_date + pd.Timedelta(days=1)
)

print(
    "Segmentation reference date:",
    segmentation_reference_date
)

Segmentation reference date: 2011-12-10 12:50:00


In [9]:
segmentation_customers = purchases["customer_id"].unique()

print(
    f"Customers with at least one valid purchase: "
    f"{len(segmentation_customers):,}"
)

Customers with at least one valid purchase: 5,852


In [10]:
all_clean_customers = set(df["customer_id"])
purchase_customers = set(purchases["customer_id"])

cancellation_only_customers = (
    all_clean_customers - purchase_customers
)

print(
    f"Customers appearing only in cancellations: "
    f"{len(cancellation_only_customers):,}"
)

Customers appearing only in cancellations: 23


## RFM analysis definitions

For full-history customer segmentation, a common segmentation reference date is
defined as one day after the most recent valid purchase in the dataset.

Recency is the number of days between this reference date and each customer's
most recent valid purchase.

Frequency is the number of distinct valid purchase invoices. We do not use the
number of transaction rows as frequency because one invoice can contain many
product lines. A customer buying 20 different products in one order has made
one purchase occasion, not 20 purchases. Cancellations therefore do not count
toward purchase frequency or recency.

Monetary value is initially defined as the sum of valid merchandise purchase
line revenue, representing gross merchandise purchase value.

In simpler terms:

**R = days since the customer's last valid purchase**

**F = number of distinct valid purchase invoices**

**M = gross valid merchandise purchase value**

## Core RFM feature construction

The valid purchase transactions are aggregated from transaction level to
customer level.

For each customer:

- **Recency** measures the number of days since the customer's most recent
  valid purchase relative to the common segmentation reference date.
- **Frequency** measures the number of distinct valid purchase invoices.
- **Monetary value** measures total gross merchandise purchase value.

Cancellation transactions are excluded from the core RFM measures and will be
represented separately through return-behaviour features.

In [11]:
rfm = (
    purchases
    .groupby("customer_id")
    .agg(
        last_purchase_date=("invoice_date", "max"),
        frequency=("invoice", "nunique"),
        monetary_value=("line_revenue", "sum")
    )
    .reset_index()
)

rfm["recency_days"] = (
    segmentation_reference_date - rfm["last_purchase_date"]
).dt.days

In [12]:
rfm = rfm[
    [
        "customer_id",
        "recency_days",
        "frequency",
        "monetary_value",
        "last_purchase_date"
    ]
]

rfm.head()

,customer_id,recency_days,frequency,monetary_value,last_purchase_date
0,12346,326,4,77353.96,2011-01-18 10:01:00
1,12347,2,8,5633.32,2011-12-07 15:52:00
2,12348,75,5,1658.40,2011-09-25 13:13:00
3,12349,19,3,3678.69,2011-11-21 09:51:00
4,12350,310,1,294.40,2011-02-02 16:01:00


In [13]:
print(f"Customer rows: {len(rfm):,}")
print(f"Unique customer IDs: {rfm['customer_id'].nunique():,}")

Customer rows: 5,852
Unique customer IDs: 5,852


In [14]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5852 entries, 0 to 5851
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         5852 non-null   string        
 1   recency_days        5852 non-null   int64         
 2   frequency           5852 non-null   int64         
 3   monetary_value      5852 non-null   float64       
 4   last_purchase_date  5852 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), string(1)
memory usage: 228.7 KB


In [15]:
assert len(rfm) == purchases["customer_id"].nunique()
assert rfm["customer_id"].is_unique
assert rfm["customer_id"].notna().all()

assert (rfm["recency_days"] >= 1).all()
assert (rfm["frequency"] >= 1).all()
assert (rfm["monetary_value"] > 0).all()

print("RFM validation passed.")

RFM validation passed.


In [16]:
rfm[
    [
        "recency_days",
        "frequency",
        "monetary_value"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
recency_days,5852.0,200.198052,208.509570,1.00,25.00,95.000,379.000,739.00
frequency,5852.0,6.253418,12.749249,1.00,1.00,3.000,7.000,373.00
monetary_value,5852.0,2979.026956,14604.963744,2.95,344.89,880.375,2289.335,608821.65


In [17]:
print("Most recent customers:")
display(
    rfm.nsmallest(
        5,
        "recency_days"
    )
)

print("Highest-frequency customers:")
display(
    rfm.nlargest(
        5,
        "frequency"
    )
)

print("Highest-value customers:")
display(
    rfm.nlargest(
        5,
        "monetary_value"
    )
)

Most recent customers:


,customer_id,recency_days,frequency,monetary_value,last_purchase_date
75,12423,1,10,2301.39,2011-12-09 10:10:00
85,12433,1,10,20581.26,2011-12-09 10:02:00
169,12518,1,5,1840.89,2011-12-09 10:13:00
177,12526,1,3,1172.66,2011-12-09 12:09:00
309,12662,1,20,6374.02,2011-12-09 11:59:00


Highest-frequency customers:


,customer_id,recency_days,frequency,monetary_value,last_purchase_date
2520,14911,1,373,276654.61,2011-12-08 15:54:00
393,12748,1,322,52194.26,2011-12-09 12:20:00
5408,17841,2,211,70858.77,2011-12-08 12:07:00
2916,15311,1,207,116476.16,2011-12-09 12:00:00
730,13089,3,203,116737.86,2011-12-07 09:02:00


Highest-value customers:


,customer_id,recency_days,frequency,monetary_value,last_purchase_date
5666,18102,1,145,608821.65,2011-12-09 11:50:00
2262,14646,2,145,526751.52,2011-12-08 12:12:00
1778,14156,10,144,303578.63,2011-11-30 10:54:00
2520,14911,1,373,276654.61,2011-12-08 15:54:00
5026,17450,8,51,246973.09,2011-12-01 13:29:00


The results above show that the distributions are strongly right-skewed:

median frequency is only 3 invoices, while the maximum is 373.

median monetary value is about £880, while the maximum exceeds £608k.

median recency is 95 days, but 25% of customers have not purchased for at least 379 days.

## Additional purchase-behaviour features

Core RFM captures recency, purchase frequency and customer value, but does not
fully describe the size or breadth of purchasing behaviour.

A small number of complementary features are therefore added:

- **Total items**: total merchandise quantity purchased.
- **Unique products**: number of distinct stock codes purchased.
- **Average order value**: gross merchandise purchase value per distinct
  purchase invoice.
- **Average items per order**: merchandise units purchased per distinct
  purchase invoice.
- **Customer tenure**: number of days between the customer's first valid
  purchase and the common segmentation reference date.

The feature set is deliberately kept compact. Additional variables are only
introduced where they represent behaviour not already captured directly by
RFM.

In [18]:
purchase_features = (
    purchases
    .groupby("customer_id")
    .agg(
        first_purchase_date=("invoice_date", "min"),
        total_items=("quantity", "sum"),
        unique_products=("stock_code", "nunique")
    )
    .reset_index()
)

In [19]:
customer_features = rfm.merge(
    purchase_features,
    on="customer_id",
    how="left"
)

In [20]:
customer_features["average_order_value"] = (
    customer_features["monetary_value"]
    / customer_features["frequency"]
)

customer_features["average_items_per_order"] = (
    customer_features["total_items"]
    / customer_features["frequency"]
)

customer_features["tenure_days"] = (
    segmentation_reference_date
    - customer_features["first_purchase_date"]
).dt.days

In [21]:
customer_features = customer_features[
    [
        "customer_id",
        "recency_days",
        "frequency",
        "monetary_value",
        "total_items",
        "unique_products",
        "average_order_value",
        "average_items_per_order",
        "tenure_days",
        "first_purchase_date",
        "last_purchase_date"
    ]
]

customer_features.head()

,customer_id,recency_days,frequency,monetary_value,total_items,unique_products,average_order_value,average_items_per_order,tenure_days,first_purchase_date,last_purchase_date
0,12346,326,4,77353.96,74240,26,19338.490,18560.000000,722,2009-12-18 10:55:00,2011-01-18 10:01:00
1,12347,2,8,5633.32,3286,126,704.165,410.750000,404,2010-10-31 14:20:00,2011-12-07 15:52:00
2,12348,75,5,1658.40,2704,24,331.680,540.800000,438,2010-09-27 14:59:00,2011-09-25 13:13:00
3,12349,19,3,3678.69,1621,137,1226.230,540.333333,589,2010-04-29 13:20:00,2011-11-21 09:51:00
4,12350,310,1,294.40,196,16,294.400,196.000000,310,2011-02-02 16:01:00,2011-02-02 16:01:00


In [22]:
assert len(customer_features) == 5852
assert customer_features["customer_id"].is_unique

assert (customer_features["total_items"] > 0).all()
assert (customer_features["unique_products"] >= 1).all()
assert (customer_features["average_order_value"] > 0).all()
assert (customer_features["average_items_per_order"] > 0).all()
assert (customer_features["tenure_days"] >= customer_features["recency_days"]).all()

print("Purchase feature validation passed.")

Purchase feature validation passed.


In [23]:
customer_features[
    [
        "total_items",
        "unique_products",
        "average_order_value",
        "average_items_per_order",
        "tenure_days"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
total_items,5852.0,1826.788961,8977.441935,1.00,192.00000,493.000000,1379.000000,367072.00
unique_products,5852.0,82.181476,116.501758,1.00,19.00000,45.000000,103.000000,2546.00
average_order_value,5852.0,389.811840,1235.883372,2.95,180.63375,283.705000,417.970000,84236.25
average_items_per_order,5852.0,257.293243,1449.712783,1.00,94.43750,158.844907,264.035714,87167.00
tenure_days,5852.0,474.005126,223.215848,1.00,312.00000,529.000000,667.000000,739.00


## Cancellation and return behaviour features

Identifiable cancellation transactions are retained separately from purchases
and are used to construct customer-level return behaviour features.

The objective is not to treat cancellations as purchases, but to capture whether
a customer frequently reverses transactions and how large those reversals are
relative to their purchase activity.

The following features are constructed:

- **Cancellation count**: number of distinct cancellation invoices.
- **Cancellation value**: absolute value of merchandise value cancelled.
- **Cancelled items**: absolute quantity associated with cancellation rows.
- **Cancellation rate**: cancellation invoices relative to total purchase
  invoices.
- **Cancellation value rate**: cancelled merchandise value relative to gross
  purchase monetary value.

Customers without identifiable cancellations receive zero values for these
features.

In [24]:
cancellation_features = (
    cancellations
    .groupby("customer_id")
    .agg(
        cancellation_count=("invoice", "nunique"),
        cancellation_value=("line_revenue", lambda x: -x.sum()),
        cancelled_items=("quantity", lambda x: -x.sum())
    )
    .reset_index()
)

In [25]:
customer_features = customer_features.merge(
    cancellation_features,
    on="customer_id",
    how="left"
)

In [26]:
cancellation_columns = [
    "cancellation_count",
    "cancellation_value",
    "cancelled_items"
]

customer_features[cancellation_columns] = (
    customer_features[cancellation_columns]
    .fillna(0)
)

In [27]:
customer_features["cancellation_rate"] = (
    customer_features["cancellation_count"]
    / customer_features["frequency"]
)

customer_features["cancellation_value_rate"] = (
    customer_features["cancellation_value"]
    / customer_features["monetary_value"]
)

In [28]:
customer_features["net_merchandise_value"] = (
    customer_features["monetary_value"]
    - customer_features["cancellation_value"]
)

In [29]:
assert (customer_features["cancellation_count"] >= 0).all()
assert (customer_features["cancellation_value"] >= 0).all()
assert (customer_features["cancelled_items"] >= 0).all()

assert customer_features[
    [
        "cancellation_count",
        "cancellation_value",
        "cancelled_items",
        "cancellation_rate",
        "cancellation_value_rate"
    ]
].notna().all().all()

print("Cancellation feature validation passed.")

Cancellation feature validation passed.


In [30]:
customer_features[
    [
        "cancellation_count",
        "cancellation_value",
        "cancelled_items",
        "cancellation_rate",
        "cancellation_value_rate",
        "net_merchandise_value"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
cancellation_count,5852.0,1.240089,3.330681,0.00,0.00,0.00,1.000000,93.000000
cancellation_value,5852.0,122.738756,2505.611910,0.00,0.00,0.00,27.750000,168469.600000
cancelled_items,5852.0,81.600991,1885.290895,0.00,0.00,0.00,8.000000,87919.000000
cancellation_rate,5852.0,0.179774,0.315697,0.00,0.00,0.00,0.250000,4.000000
cancellation_value_rate,5852.0,0.024406,0.101760,0.00,0.00,0.00,0.014631,2.858386
net_merchandise_value,5852.0,2856.288200,14125.530438,-1343.24,336.07,861.96,2225.182500,606243.250000


In [31]:
print(
    "Customers with at least one cancellation:",
    (customer_features["cancellation_count"] > 0).sum()
)

print(
    "Customers with no cancellations:",
    (customer_features["cancellation_count"] == 0).sum()
)

Customers with at least one cancellation: 2422
Customers with no cancellations: 3430


## Feature distribution and redundancy assessment

Before selecting variables for customer segmentation, the engineered features
are examined for extreme skewness, unusual ratios and redundancy.

Large values are not automatically treated as errors. In transactional retail
data, high-frequency and high-value customers may represent genuine and
strategically important customer behaviour.

The objective of this assessment is therefore not to remove statistical
outliers simply because we can, but to identify heavily skewed features that may require transformation before clustering, unusual but potentially valid customer behaviour, strongly redundant features that could otherwise give the same behavioural dimension excessive weight in distance-based clustering.

In [32]:
print(
    "Customers with cancellation_rate > 1:",
    (customer_features["cancellation_rate"] > 1).sum()
)

print(
    "Customers with cancellation_value_rate > 1:",
    (customer_features["cancellation_value_rate"] > 1).sum()
)

print(
    "Customers with negative net merchandise value:",
    (customer_features["net_merchandise_value"] < 0).sum()
)

Customers with cancellation_rate > 1: 62
Customers with cancellation_value_rate > 1: 10
Customers with negative net merchandise value: 10


In [33]:
customer_features[
    (customer_features["cancellation_rate"] > 1) |
    (customer_features["cancellation_value_rate"] > 1) |
    (customer_features["net_merchandise_value"] < 0)
].sort_values(
    "cancellation_value_rate",
    ascending=False
).head(20)

,customer_id,recency_days,frequency,monetary_value,total_items,unique_products,average_order_value,average_items_per_order,tenure_days,first_purchase_date,last_purchase_date,cancellation_count,cancellation_value,cancelled_items,cancellation_rate,cancellation_value_rate,net_merchandise_value
3532,15935,239,1,108.04,114,11,108.040,114.0,239,2011-04-14 13:27:00,2011-04-14 13:27:00,1.0,308.82,286.0,1.000000,2.858386,-2.007800e+02
732,13091,28,2,1108.80,1338,40,554.400,669.0,730,2009-12-10 12:44:00,2011-11-11 15:54:00,3.0,2452.04,2424.0,1.500000,2.211436,-1.343240e+03
1835,14213,405,1,1192.20,244,5,1192.200,244.0,405,2010-10-31 10:37:00,2010-10-31 10:37:00,1.0,2384.40,488.0,1.000000,2.000000,-1.192200e+03
3844,16252,383,1,295.09,158,21,295.090,158.0,383,2010-11-22 11:57:00,2010-11-22 11:57:00,1.0,590.18,316.0,1.000000,2.000000,-2.950900e+02
752,13112,541,1,20.60,11,2,20.600,11.0,541,2010-06-16 17:03:00,2010-06-16 17:03:00,2.0,26.04,12.0,2.000000,1.264078,-5.440000e+00
255,12607,60,1,1579.51,1228,101,1579.510,1228.0,60,2011-10-10 16:06:00,2011-10-10 16:06:00,1.0,1579.51,1228.0,1.000000,1.000000,-2.273737e-13
208,12558,8,1,269.96,196,11,269.960,196.0,8,2011-12-02 10:41:00,2011-12-02 10:41:00,1.0,269.96,196.0,1.000000,1.000000,-5.684342e-14
5838,18274,30,1,175.92,88,11,175.920,88.0,30,2011-11-09 17:03:00,2011-11-09 17:03:00,1.0,175.92,88.0,1.000000,1.000000,-2.842171e-14
2173,14557,85,1,788.38,510,16,788.380,510.0,85,2011-09-15 15:48:00,2011-09-15 15:48:00,1.0,788.38,510.0,1.000000,1.000000,-1.136868e-13
998,13364,71,1,134.96,71,10,134.960,71.0,71,2011-09-29 18:13:00,2011-09-29 18:13:00,1.0,134.96,71.0,1.000000,1.000000,-2.842171e-14


In [34]:
# Examine customer skewness

feature_columns = [
    "recency_days",
    "frequency",
    "monetary_value",
    "total_items",
    "unique_products",
    "average_order_value",
    "average_items_per_order",
    "tenure_days",
    "cancellation_count",
    "cancellation_value",
    "cancelled_items",
    "cancellation_rate",
    "cancellation_value_rate",
    "net_merchandise_value"
]

In [35]:
skewness = (
    customer_features[feature_columns]
    .skew()
    .sort_values(ascending=False)
)

skewness

,0
cancellation_value,57.211151
average_order_value,54.982886
average_items_per_order,45.997534
cancelled_items,41.508026
net_merchandise_value,27.071000
monetary_value,25.583767
total_items,21.535901
cancellation_value_rate,12.320906
frequency,12.040277
cancellation_count,11.084205


In [36]:
correlation_matrix = customer_features[
    feature_columns
].corr(method="spearman")

correlation_matrix.round(2)

,recency_days,frequency,monetary_value,total_items,unique_products,average_order_value,average_items_per_order,tenure_days,cancellation_count,cancellation_value,cancelled_items,cancellation_rate,cancellation_value_rate,net_merchandise_value
recency_days,1.00,-0.56,-0.51,-0.51,-0.47,-0.17,-0.19,0.15,-0.31,-0.28,-0.28,-0.18,-0.20,-0.51
frequency,-0.56,1.00,0.86,0.82,0.72,0.21,0.18,0.50,0.58,0.55,0.55,0.37,0.43,0.86
monetary_value,-0.51,0.86,1.00,0.95,0.78,0.65,0.54,0.42,0.58,0.56,0.56,0.41,0.43,0.99
total_items,-0.51,0.82,0.95,1.00,0.76,0.61,0.68,0.39,0.55,0.53,0.53,0.38,0.39,0.94
unique_products,-0.47,0.72,0.78,0.76,1.00,0.43,0.41,0.35,0.45,0.42,0.42,0.30,0.30,0.78
average_order_value,-0.17,0.21,0.65,0.61,0.43,1.00,0.82,0.09,0.28,0.28,0.28,0.25,0.20,0.65
average_items_per_order,-0.19,0.18,0.54,0.68,0.41,0.82,1.00,0.04,0.22,0.22,0.23,0.19,0.14,0.54
tenure_days,0.15,0.50,0.42,0.39,0.35,0.09,0.04,1.00,0.34,0.33,0.32,0.22,0.25,0.42
cancellation_count,-0.31,0.58,0.58,0.55,0.45,0.28,0.22,0.34,1.00,0.97,0.96,0.93,0.93,0.56
cancellation_value,-0.28,0.55,0.56,0.53,0.42,0.28,0.22,0.33,0.97,1.00,0.99,0.92,0.96,0.53


In [37]:
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

strong_correlations = (
    upper_triangle
    .stack()
    .sort_values(
        key=abs,
        ascending=False
    )
)

strong_correlations.head(20)

monetary_value       net_merchandise_value      0.994843
cancellation_value   cancelled_items            0.986077
cancellation_count   cancellation_value         0.968510
                     cancelled_items            0.964130
cancellation_value   cancellation_value_rate    0.963699
cancellation_rate    cancellation_value_rate    0.958112
cancelled_items      cancellation_value_rate    0.952145
monetary_value       total_items                0.946258
total_items          net_merchandise_value      0.942130
cancellation_count   cancellation_rate          0.931830
                     cancellation_value_rate    0.926005
cancellation_value   cancellation_rate          0.919888
cancelled_items      cancellation_rate          0.918157
frequency            monetary_value             0.860091
                     net_merchandise_value      0.858201
average_order_value  average_items_per_order    0.819807
frequency            total_items                0.819235
unique_products      net_merchandise_value      0.779706
monetary_value       unique_products            0.775606
total_items          unique_products            0.764771
dtype: float64

In [38]:
segmentation_feature_columns = [
    "recency_days",
    "frequency",
    "monetary_value",
    "unique_products",
    "average_order_value",
    "tenure_days",
    "cancellation_value_rate"
]

## Feature-selection implications

The engineered customer features show substantial right-skew and several
strong relationships between variables.

Gross monetary value and net merchandise value are almost perfectly correlated,
while total purchased quantity is also strongly associated with monetary value.
Similarly, cancellation count, cancellation value and cancelled quantity form a
highly correlated group.

Including all engineered variables in a distance-based clustering model would
therefore give some behavioural dimensions disproportionate influence simply
because they are represented by several related variables.

The complete customer feature table is retained for business interpretation and
cluster profiling, but a smaller subset is selected as the initial segmentation
input:

- `recency_days` — current customer engagement;
- `frequency` — number of distinct purchase occasions;
- `monetary_value` — gross merchandise customer value;
- `unique_products` — breadth of purchasing behaviour;
- `average_order_value` — typical financial size of an order;
- `tenure_days` — length of the observed customer relationship; and
- `cancellation_value_rate` — cancelled merchandise value relative to gross
  purchase value.

Extreme values are not removed at this stage. High-value and high-frequency
customers may represent genuine commercially important behaviour rather than
data errors.

The selected clustering variables are highly skewed and will therefore require
appropriate transformation and scaling before distance-based clustering. Those
operations are intentionally left for the segmentation modelling phase so
that this notebook preserves the original, interpretable customer features.

In [39]:
assert len(customer_features) == 5852
assert customer_features["customer_id"].is_unique
assert customer_features["customer_id"].notna().all()

assert customer_features[
    segmentation_feature_columns
].notna().all().all()

print("Final customer feature table validation passed.")

Final customer feature table validation passed.


In [40]:
CUSTOMER_FEATURE_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "customer_features.parquet"
)

CUSTOMER_FEATURE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

customer_features.to_parquet(
    CUSTOMER_FEATURE_PATH,
    index=False
)

print(f"Saved to: {CUSTOMER_FEATURE_PATH}")

Saved to: /content/data/processed/customer_features.parquet


In [41]:
test_features = pd.read_parquet(CUSTOMER_FEATURE_PATH)

assert len(test_features) == len(customer_features)
assert list(test_features.columns) == list(customer_features.columns)

print("Customer feature Parquet round-trip validation passed.")

Customer feature Parquet round-trip validation passed.
